In [ ]:
# Install vLLM + utilities
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

In [ ]:
# Clone repo to access pre-built splits
REPO_URL = "https://github.com/SpaceSpiff20/daniel-splits.git"
REPO_DIR = "daniel-splits"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git checkout daniel_splits
!git pull

In [ ]:
!ls data/splits_out/superglue_boolq | head -20

In [ ]:
import os
import re
import json
import time
import numpy as np
from pathlib import Path
from typing import List, Dict
from datasets import load_dataset

print("Loading SuperGLUE BoolQ...")
ds = load_dataset("super_glue", "boolq", split="validation")

print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
print(f"Label distribution: {dict(zip(*np.unique([str(x) for x in ds['label']], return_counts=True)))}")

In [ ]:
# --- Config ---
MODEL_NAME     = "Qwen/Qwen3-4B"
MAX_NEW_TOKENS = 8
EVAL_SIZE      = len(ds)        # set to e.g. 50 for a quick smoke test
ENABLE_THINKING = False         # recommended False for fast classification
N_VALUES       = [16, 32, 64, 128, 256]
SEEDS          = [0]
SPLITS_DIR     = Path("data/splits_out/superglue_boolq")

SYSTEM_PROMPT = (
    "You are a yes/no question answering system. "
    "Given a passage and a question, answer with exactly one word: Yes or No."
)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "enable_thinking": ENABLE_THINKING,
    "N_values": N_VALUES,
    "seeds": SEEDS,
})

In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

print("Loading Qwen3-4B with vLLM...")
print("  PagedAttention: enabled (automatic)")
print("  Cont. batching: enabled (automatic)")
print("  Prefix caching: enabled")

llm = LLM(
    model=MODEL_NAME,
    dtype="float16",
    gpu_memory_utilization=0.95,
    max_model_len=32768,
    enforce_eager=False,
    enable_prefix_caching=True,
)

sampling_params = SamplingParams(
    temperature=0,
    top_k=20,
    max_tokens=MAX_NEW_TOKENS,
)

print("Model loaded successfully!")

In [ ]:
# --- Output directory (Colab Drive if available) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/Qwen3_4B_BoolQ_ICL_vLLM"
except Exception:
    DRIVE_SAVE_DIR = os.path.abspath("./qwen3_4b_boolq_icl_vllm_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

In [ ]:
# --- Helper functions ---

def read_jsonl(path: Path) -> List[Dict]:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def label_to_yesno(label: str) -> str:
    """Convert split label (True/False/0/1) to yes/no string for prompt."""
    s = str(label).strip().lower()
    if s in {"true", "1", "yes"}:
        return "yes"
    if s in {"false", "0", "no"}:
        return "no"
    raise ValueError(f"Unknown label: {label}")


def parse_x(x: str):
    """Extract passage and question from split x field: 'Passage: ...\nQuestion: ...'."""
    parts = x.split("\nQuestion:")
    if len(parts) == 2:
        passage = parts[0].replace("Passage:", "").strip()
        question = parts[1].strip()
        return passage, question
    return x.strip(), ""


def build_icl_prompt(kshot: List[Dict], passage: str, question: str) -> str:
    """Build a chat-template ICL prompt with k-shot BoolQ examples."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]

    for ex in kshot:
        dp, dq = parse_x(ex["x"])
        msgs.append({"role": "user",      "content": f"Passage: {dp}\nQuestion: {dq}?\nAnswer:"})
        msgs.append({"role": "assistant", "content": label_to_yesno(ex["label"])})

    msgs.append({"role": "user", "content": f"Passage: {passage}\nQuestion: {question}?\nAnswer:"})

    return tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )


def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text


def extract_bool(text: str):
    """Map model output to True/False. Returns None if unparseable."""
    text = strip_thinking(text).strip().lower()
    if text.startswith("yes"):
        return True
    if text.startswith("no"):
        return False
    if "yes" in text and "no" not in text:
        return True
    if "no" in text and "yes" not in text:
        return False
    if "true" in text:
        return True
    if "false" in text:
        return False
    return None


print("Helper functions defined.")

In [ ]:
# --- Sanity-check: print an example ICL prompt ---
sample_kshot = read_jsonl(SPLITS_DIR / "N16_seed0.jsonl")
sample_prompt = build_icl_prompt(sample_kshot, ds["passage"][0], ds["question"][0])
print(f"N=16 prompt length (chars): {len(sample_prompt)}")
print("\nFirst 1000 chars:")
print(sample_prompt[:1000])

In [ ]:
# --- Run ICL evaluation across N values and seeds ---
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

passages  = ds["passage"]
questions = ds["question"]
labels    = ds["label"]

eval_passages  = passages[:EVAL_SIZE]
eval_questions = questions[:EVAL_SIZE]
eval_labels    = labels[:EVAL_SIZE]

all_results = []

for N in N_VALUES:
    for seed in SEEDS:
        split_path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
        kshot = read_jsonl(split_path)

        print(f"\n{'='*60}")
        print(f"N={N}, seed={seed} — building {len(eval_passages)} prompts...")

        prompts = [
            build_icl_prompt(kshot, p, q)
            for p, q in zip(eval_passages, eval_questions)
        ]

        avg_prompt_len = int(np.mean([len(tokenizer.encode(p)) for p in prompts[:50]]))
        print(f"Avg prompt tokens (sample of 50): ~{avg_prompt_len}")

        print(f"Generating responses with vLLM...")
        gen_start = time.time()
        outputs = llm.generate(prompts, sampling_params)
        gen_time = time.time() - gen_start

        total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        throughput = total_new_tokens / gen_time if gen_time > 0 else None

        print(f"\nGeneration complete.")
        print(f"  Time:       {gen_time/60:.1f} min")
        print(f"  Tokens:     {total_new_tokens:,}")
        if throughput:
            print(f"  Throughput: {throughput:.1f} tokens/sec")

        # --- Score ---
        correct = 0
        unknown = 0
        scored = []
        for i, o in enumerate(outputs):
            response = o.outputs[0].text
            pred = extract_bool(response)
            gt   = bool(eval_labels[i])
            is_correct = (pred == gt) if pred is not None else False
            if pred is None:
                unknown += 1
            if is_correct:
                correct += 1
            scored.append({
                "idx":         i,
                "ground_truth": gt,
                "prediction":  pred,
                "raw_response": response.strip(),
                "is_correct":  is_correct,
                "is_unknown":  pred is None,
            })

        accuracy     = correct / len(scored) if scored else 0
        unknown_frac = unknown / len(scored) if scored else 0

        result = {
            "method":             f"icl_vllm_thinking={ENABLE_THINKING}",
            "model":              MODEL_NAME,
            "dataset":            "super_glue/boolq:validation",
            "N":                  N,
            "seed":               seed,
            "eval_size":          len(scored),
            "accuracy":           accuracy,
            "unknown_frac":       unknown_frac,
            "total_new_tokens":   total_new_tokens,
            "gen_tokens_per_sec": throughput,
            "total_gen_time_min": gen_time / 60,
            "avg_prompt_tokens":  avg_prompt_len,
        }
        all_results.append(result)

        print(f"\n  accuracy={accuracy:.4f}  unknown={unknown_frac:.4f}  time={gen_time/60:.1f}min")

        # Save per-run scored outputs
        out_file = os.path.join(DRIVE_SAVE_DIR, f"boolq_icl_N{N}_seed{seed}_scored.json")
        with open(out_file, "w") as f:
            json.dump(scored, f)
        print(f"  Scored outputs saved to: {out_file}")

print("\nAll runs complete.")

In [ ]:
# --- Summary table ---
import pandas as pd

df = pd.DataFrame(all_results).sort_values(["N", "seed"])

display(df[[
    "N",
    "seed",
    "accuracy",
    "unknown_frac",
    "gen_tokens_per_sec",
    "total_gen_time_min",
    "avg_prompt_tokens",
]])

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "icl_results.json")
with open(METRICS_FILE, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")

In [ ]:
# --- Plots ---
import matplotlib.pyplot as plt

df_plot = df.groupby("N").agg({"accuracy": "mean", "gen_tokens_per_sec": "mean", "avg_prompt_tokens": "mean"}).reset_index()

# Plot 1: Accuracy vs N
plt.figure(figsize=(8, 4))
plt.plot(df_plot["N"], df_plot["accuracy"], marker="o")
plt.xlabel("N (Number of In-Context Examples)")
plt.ylabel("Accuracy")
plt.title("ICL Accuracy vs N (BoolQ, Qwen3-4B, vLLM)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Generation Speed vs N
plt.figure(figsize=(8, 4))
plt.plot(df_plot["N"], df_plot["gen_tokens_per_sec"], marker="o", color="orange")
plt.xlabel("N (Number of In-Context Examples)")
plt.ylabel("Generation Tokens / Second")
plt.title("ICL Generation Speed vs N")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 3: Prompt Length vs N
plt.figure(figsize=(8, 4))
plt.plot(df_plot["N"], df_plot["avg_prompt_tokens"], marker="o", color="green")
plt.xlabel("N (Number of In-Context Examples)")
plt.ylabel("Average Prompt Tokens")
plt.title("Prompt Length Growth vs N")
plt.grid(True)
plt.tight_layout()
plt.show()